> **Nivel:** Programadores con experiencia en R (básico-intermedio)  
> **Capítulo:** 6 — Diccionarios  
> **Prerrequisitos:** Capítulos 1, 3, 4 y 5 (variables, listas, loops, condicionales)  
> **Tiempo estimado:** 120–150 minutos

---

## 🎯 Objetivos del Capítulo

Al terminar este capítulo serás capaz de:

- Crear y acceder a diccionarios con distintas sintaxis
- Agregar, modificar y eliminar pares clave-valor de forma segura
- Recorrer diccionarios iterando sobre claves, valores o ambos
- Usar dict comprehensions para crear diccionarios en una línea
- Trabajar con diccionarios anidados y listas de diccionarios
- Entender cuándo un diccionario es mejor que una lista en Python
- Mapear los patrones de R (`list()`, `names()`, data frames) a diccionarios de Python

---

## 🗺️ Mapa del Capítulo

```
6.1  ¿Qué es un diccionario? — Comparación con R
6.2  Crear y acceder a diccionarios
     └── Acceso directo y .get()
     └── Claves, valores y pares
6.3  Modificar diccionarios
     └── Agregar, actualizar, eliminar
     └── Métodos: update(), pop(), setdefault()
6.4  Recorrer diccionarios
     └── .items(), .keys(), .values()
     └── Ordenar al recorrer
6.5  Dict Comprehensions ⭐
     └── Transformar y filtrar
     └── Invertir diccionarios
6.6  Diccionarios anidados
     └── Dict de dicts
     └── Lista de dicts (datos tabulares)
6.7  Patrones avanzados
     └── defaultdict, Counter
     └── Combinar diccionarios
6.8  Errores comunes
✏️   Ejercicios Graduales (3 niveles)
🚀   Mini Proyecto
🏆   Desafíos
```

---

## 🔄 Panorama General: Diccionarios Python vs R

| Estructura Python | Equivalente R | Diferencias clave |
|---|---|---|
| `{'a': 1, 'b': 2}` | `list(a=1, b=2)` | Python no usa `$`; acceso con `['clave']` |
| `d['clave']` | `l$nombre` o `l[['nombre']]` | `$` no existe en Python |
| `d.get('k', default)` | `l[['k']] %||% default` (con rlang) | `.get()` es nativo |
| `d.keys()` | `names(l)` | Retorna vista dinámica |
| `d.values()` | `unname(l)` | Retorna vista dinámica |
| `d.items()` | No directo | Pares `(clave, valor)` |
| `{k: v for k, v in ...}` | `setNames(vals, keys)` | Dict comprehension |
| Lista de dicts | `data.frame()` | Formato de datos tabulares |
| `dict` anidado | `list()` anidado | Acceso encadenado `d['a']['b']` |

::: {.callout-tip title="💡 Mentalidad Clave"}
En R, la estructura **`list()`** nombrada es el equivalente más cercano al diccionario de Python.
Sin embargo, hay diferencias importantes:

- En R usas `l$nombre` o `l[["nombre"]]` para acceder. En Python solo `d["nombre"]`.
- En R `names(l)` modifica nombres in-place. En Python `d.keys()` retorna una *vista*.
- Los data frames de R (filas = observaciones, columnas = variables) se representan en Python como **listas de diccionarios** o, más idiomáticamente, con **pandas DataFrames**.

El diccionario de Python está optimizado para búsqueda `O(1)` — buscar un valor por clave es igual de rápido sin importar cuántos elementos haya.
:::

---

## 6.1 ¿Qué es un Diccionario?

Un diccionario es una colección de **pares clave-valor** donde cada clave es única. Es la estructura de datos más versátil de Python después de las listas.

In [ ]:
# Un diccionario se define con llaves {}
# Formato: {clave: valor, clave: valor, ...}

persona = {
    'nombre':   'Ana García',
    'edad':     28,
    'ciudad':   'CDMX',
    'activo':   True,
    'lenguajes':['Python', 'R', 'SQL']   # el valor puede ser cualquier tipo
}

print(persona)
print(type(persona))   # <class 'dict'>
print(f"Campos: {len(persona)}")

In [ ]:
# Las claves pueden ser cualquier tipo INMUTABLE
# (strings, enteros, tuplas — no listas)

d = {
    'texto':      'clave string',
    42:           'clave entero',
    (1, 2):       'clave tupla',
    True:         'clave booleana',
}

print(d['texto'])
print(d[42])
print(d[(1, 2)])

# Las claves son únicas — la última asignación gana
repetida = {'a': 1, 'b': 2, 'a': 99}   # 'a' aparece dos veces
print(f"\nClave repetida: {repetida}")   # {'a': 99, 'b': 2}

# Diccionario vacío
vacio = {}
vacio2 = dict()   # forma alternativa
print(f"Tipos: {type(vacio)}, {type(vacio2)}")

::: {.callout-note title="R vs Python — Crear estructuras clave-valor"}
```r
# En R — lista nombrada
persona <- list(
  nombre   = 'Ana García',
  edad     = 28,
  ciudad   = 'CDMX',
  activo   = TRUE,
  lenguajes = c('Python', 'R', 'SQL')
)

# También con setNames()
d <- setNames(list(1, 2, 3), c('a', 'b', 'c'))
```
```python
# En Python — diccionario
persona = {
    'nombre':    'Ana García',
    'edad':      28,
    'ciudad':    'CDMX',
    'activo':    True,
    'lenguajes': ['Python', 'R', 'SQL']
}

# También con dict() constructor
d = dict(a=1, b=2, c=3)      # ← claves sin comillas aquí
d = dict(zip(['a','b','c'], [1, 2, 3]))
```
:::

---

## 6.2 Acceso a Valores

### 6.2.1 Acceso directo y `.get()`

In [ ]:
persona = {
    'nombre': 'Carlos Ramírez',
    'edad':   32,
    'ciudad': 'Guadalajara',
    'email':  'carlos@mail.com'
}

# Acceso directo — lanza KeyError si la clave no existe
print(persona['nombre'])    # 'Carlos Ramírez'
print(persona['edad'])      # 32

# KeyError si la clave no existe:
try:
    print(persona['telefono'])
except KeyError as e:
    print(f"KeyError: {e} — la clave no existe")

# .get() — acceso seguro, retorna None (o default) si no existe
print(persona.get('telefono'))            # None  — sin error
print(persona.get('telefono', 'N/A'))     # 'N/A' — valor por defecto
print(persona.get('email', 'Sin email'))  # 'carlos@mail.com' — sí existe

::: {.callout-note title="R vs Python — Acceso a elementos"}
```r
# En R — múltiples formas
persona <- list(nombre='Ana', edad=28, ciudad='CDMX')

persona$nombre           # 'Ana'  — con $
persona[['nombre']]      # 'Ana'  — con [[ ]]
persona['nombre']        # lista de un elemento — con [ ]

# Acceso seguro (si no existe):
persona$telefono         # NULL (sin error)
persona[['telefono']]    # NULL (sin error)
```
```python
# En Python — solo una forma
persona = {'nombre': 'Ana', 'edad': 28, 'ciudad': 'CDMX'}

persona['nombre']        # 'Ana'  — acceso directo
# persona.nombre         # ❌ AttributeError — no existe el operador $

# Acceso seguro:
persona.get('telefono')         # None (sin error)
persona.get('telefono', 'N/A')  # 'N/A' con valor por defecto
```

| Acceso | R | Python |
|---|---|---|
| Seguro (sin error) | `l$clave` → NULL | `d.get('clave')` → None |
| Error si no existe | `l[['clave']]` → NULL | `d['clave']` → KeyError |
| Con default | `l$clave %||% 'default'` | `d.get('clave', 'default')` |
:::

### 6.2.2 Vistas: claves, valores y pares

In [ ]:
inventario = {
    'manzanas': 50,
    'naranjas':  30,
    'peras':     45,
    'uvas':     120,
    'mangos':    18
}

# .keys()   — vista de todas las claves
# .values() — vista de todos los valores
# .items()  — vista de todos los pares (clave, valor)

claves  = inventario.keys()
valores = inventario.values()
pares   = inventario.items()

print(f"Claves:  {list(claves)}")
print(f"Valores: {list(valores)}")
print(f"Pares:   {list(pares)}")

# Las vistas son DINÁMICAS — se actualizan automáticamente
print(f"\nAntes de agregar: {len(claves)} claves")
inventario['kiwis'] = 35
print(f"Después de agregar: {len(claves)} claves")   # ya incluye 'kiwis'

# Operaciones sobre las vistas
print(f"\n¿Hay manzanas?  {'manzanas' in inventario}")
print(f"Total frutas:   {len(inventario)}")
print(f"Total unidades: {sum(inventario.values())}")
print(f"Promedio:       {sum(inventario.values())/len(inventario):.1f}")

::: {.callout-note title="R vs Python — Claves y valores"}
```r
# En R
inventario <- list(manzanas=50, naranjas=30, peras=45)
names(inventario)          # claves: c('manzanas', 'naranjas', 'peras')
unlist(inventario)         # valores como vector nombrado
length(inventario)         # número de pares
'manzanas' %in% names(inventario)  # pertenencia de clave
sum(unlist(inventario))    # suma de valores
```
```python
# En Python
inventario = {'manzanas': 50, 'naranjas': 30, 'peras': 45}
list(inventario.keys())    # claves
list(inventario.values())  # valores
len(inventario)            # número de pares
'manzanas' in inventario   # pertenencia de clave (más simple)
sum(inventario.values())   # suma de valores
```
⚠️ En Python, `'clave' in diccionario` verifica claves, **no valores**. Para valores: `'valor' in diccionario.values()`.
:::

---

## 6.3 Modificar Diccionarios

### 6.3.1 Agregar y actualizar

In [ ]:
config = {'tema': 'oscuro', 'idioma': 'español', 'volumen': 70}
print(f"Original: {config}")

# Agregar nueva clave (misma sintaxis que modificar)
config['notificaciones'] = True
config['fuente_tamano']  = 14
print(f"Con nuevas claves: {config}")

# Modificar valor existente
config['volumen'] = 85
print(f"Volumen cambiado: {config}")

# .update() — agregar/modificar múltiples pares a la vez
config.update({'idioma': 'inglés', 'zoom': 1.2, 'volumen': 90})
print(f"Tras update: {config}")

# .setdefault() — agrega SOLO si la clave NO existe
config.setdefault('volumen', 50)     # ya existe → no cambia
config.setdefault('brillo', 80)      # no existe → agrega con 80
print(f"\nVolumen (setdefault no cambió): {config['volumen']}")
print(f"Brillo (setdefault agregó):     {config['brillo']}")

### 6.3.2 Eliminar elementos

In [ ]:
datos = {'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5}
print(f"Original: {datos}")

# del — elimina por clave, sin retornar valor
del datos['a']
print(f"del 'a':  {datos}")

# .pop(clave) — elimina Y retorna el valor
valor_b = datos.pop('b')
print(f".pop('b') retornó: {valor_b}")
print(f"Dict tras pop:    {datos}")

# .pop(clave, default) — versión segura (no lanza KeyError)
valor_z = datos.pop('z', 'no existía')
print(f".pop('z', default): {valor_z}")  # no lanza error

# .popitem() — elimina y retorna el ÚLTIMO par insertado (Python 3.7+)
ultimo = datos.popitem()
print(f".popitem() retornó: {ultimo}")
print(f"Dict tras popitem: {datos}")

# .clear() — vacía el diccionario
datos.clear()
print(f".clear():  {datos}")

::: {.callout-note title="R vs Python — Modificar listas/diccionarios"}
```r
# En R — listas nombradas
config <- list(tema='oscuro', volumen=70)

# Agregar / modificar
config$nuevaClave <- 'valor'
config[['volumen']] <- 85

# Eliminar
config$tema <- NULL           # asignar NULL elimina el elemento
config[['tema']] <- NULL

# Actualizar múltiples (con modifyList)
config <- modifyList(config, list(volumen=90, zoom=1.2))
```
```python
# En Python — diccionarios
config = {'tema': 'oscuro', 'volumen': 70}

# Agregar / modificar
config['nuevaClave'] = 'valor'
config['volumen'] = 85

# Eliminar
del config['tema']            # sin retornar valor
valor = config.pop('tema')    # retornando el valor

# Actualizar múltiples
config.update({'volumen': 90, 'zoom': 1.2})
```

| Operación | R | Python |
|---|---|---|
| Agregar | `l$nueva <- v` | `d['nueva'] = v` |
| Eliminar | `l$clave <- NULL` | `del d['clave']` |
| Eliminar + obtener | No directo | `d.pop('clave')` |
| Solo si no existe | No directo | `d.setdefault('k', v)` |
| Actualizar múltiples | `modifyList(l, l2)` | `d.update(d2)` |
:::

---

## 6.4 Recorrer Diccionarios

### 6.4.1 Iterar sobre pares, claves y valores

In [ ]:
capitales = {
    'México':    'Ciudad de México',
    'Colombia':  'Bogotá',
    'Argentina': 'Buenos Aires',
    'Chile':     'Santiago',
    'Perú':      'Lima'
}

# Iterar sobre pares clave-valor — la forma más común
print("--- Iterar con .items() ---")
for pais, capital in capitales.items():
    print(f"  {pais:<12} → {capital}")

# Iterar solo sobre claves (comportamiento por defecto)
print("\n--- Iterar claves (ordenadas) ---")
for pais in sorted(capitales.keys()):
    print(f"  {pais}")

# Iterar solo sobre valores
print("\n--- Iterar valores (ordenados) ---")
for capital in sorted(capitales.values()):
    print(f"  {capital}")

In [ ]:
# Ordenar al iterar — sorted() con key
precios = {
    'Laptop':   15999,
    'Monitor':   5999,
    'Teclado':    799,
    'Mouse':      349,
    'Audífonos': 1299,
    'Webcam':     899,
}

print("--- Por precio (ascendente) ---")
for prod, precio in sorted(precios.items(), key=lambda x: x[1]):
    print(f"  {prod:<12}: ${precio:>7,}")

print("\n--- Por nombre (descendente) ---")
for prod, precio in sorted(precios.items(), key=lambda x: x[0], reverse=True):
    print(f"  {prod:<12}: ${precio:>7,}")

# Filtrar mientras se itera
print("\n--- Solo productos < $1,000 ---")
baratos = {prod: precio for prod, precio in precios.items() if precio < 1000}
for prod, precio in baratos.items():
    print(f"  {prod:<12}: ${precio:>7,}")

::: {.callout-note title="R vs Python — Iterar sobre listas nombradas"}
```r
# En R
capitales <- list(México='CDMX', Colombia='Bogotá', Argentina='Buenos Aires')

# Iterar sobre nombres y valores
for (pais in names(capitales)) {
  cat(pais, "→", capitales[[pais]], "\n")
}

# Con lapply/sapply
lapply(names(capitales), function(p) cat(p, "→", capitales[[p]], "\n"))

# Ordenar por valor
capitales[order(unlist(capitales))]
```
```python
# En Python
capitales = {'México': 'CDMX', 'Colombia': 'Bogotá', 'Argentina': 'Buenos Aires'}

# Iterar sobre pares
for pais, capital in capitales.items():
    print(f"{pais} → {capital}")

# Ordenar por valor
for pais, capital in sorted(capitales.items(), key=lambda x: x[1]):
    print(f"{pais} → {capital}")
```
El patrón `for clave, valor in d.items()` de Python es el equivalente directo de `for (n in names(l))` + `l[[n]]` de R, pero más conciso.
:::

---

## 6.5 Dict Comprehensions ⭐

Al igual que las list comprehensions, las dict comprehensions permiten crear diccionarios en una línea.

In [ ]:
# Sintaxis: {clave_expr: valor_expr for elemento in iterable}

# Crear diccionario desde listas
frutas  = ['manzana', 'plátano', 'cereza', 'durazno']
precios = [25, 15, 40, 30]

# Forma larga
frutas_precios = {}
for f, p in zip(frutas, precios):
    frutas_precios[f] = p

# Forma corta (dict comprehension)
frutas_precios = {f: p for f, p in zip(frutas, precios)}
print(f"Precios: {frutas_precios}")

# Desde range
cuadrados = {n: n**2 for n in range(1, 11)}
print(f"\nCuadrados: {cuadrados}")

# Transformar claves y valores
mayusculas = {k.upper(): v * 1.16 for k, v in frutas_precios.items()}
print(f"\nCon IVA (mayúsculas): {mayusculas}")

In [ ]:
# Dict comprehensions con filtro
precios_productos = {
    'Laptop': 15999, 'Mouse': 349, 'Monitor': 5999,
    'Teclado': 799, 'Audífonos': 1299, 'Webcam': 899
}

# Solo productos baratos
baratos = {prod: precio for prod, precio in precios_productos.items()
           if precio < 1000}
print(f"Baratos: {baratos}")

# Aplicar descuento a los caros
con_descuento = {
    prod: round(precio * 0.9, 2) if precio > 5000 else precio
    for prod, precio in precios_productos.items()
}
print(f"\nCon descuento en caros: {con_descuento}")

# Invertir un diccionario (intercambiar claves y valores)
morse_parcial = {'A': '.-', 'B': '-...', 'C': '-.-.', 'S': '...', 'O': '---'}
invertido = {v: k for k, v in morse_parcial.items()}
print(f"\nMorse original: {morse_parcial}")
print(f"Invertido:      {invertido}")

::: {.callout-note title="R vs Python — Dict comprehensions"}
```r
# En R — varias formas de crear listas nombradas transformadas
frutas  <- c('manzana', 'plátano', 'cereza')
precios <- c(25, 15, 40)

# Crear lista nombrada desde vectores
frutas_precios <- setNames(as.list(precios), frutas)

# Transformar con sapply
con_iva <- sapply(precios, function(p) p * 1.16)
names(con_iva) <- toupper(frutas)

# Invertir lista nombrada
invertido <- setNames(as.list(names(mi_lista)), unlist(mi_lista))
```
```python
# En Python — dict comprehension
frutas  = ['manzana', 'plátano', 'cereza']
precios = [25, 15, 40]

# Crear desde dos listas
frutas_precios = {f: p for f, p in zip(frutas, precios)}

# Transformar
con_iva = {f.upper(): round(p * 1.16, 2) for f, p in zip(frutas, precios)}

# Invertir
invertido = {v: k for k, v in mi_dict.items()}
```
:::

---

## 6.6 Diccionarios Anidados

### 6.6.1 Diccionario de diccionarios

In [ ]:
# Diccionarios que contienen otros diccionarios
usuarios = {
    'ana_g': {
        'nombre':    'Ana García',
        'edad':      28,
        'lenguajes': ['Python', 'R', 'SQL'],
        'nivel':     'senior',
        'activo':    True
    },
    'carlos_m': {
        'nombre':    'Carlos Martínez',
        'edad':      34,
        'lenguajes': ['Java', 'C++', 'Python'],
        'nivel':     'principal',
        'activo':    True
    },
    'bea_l': {
        'nombre':    'Beatriz López',
        'edad':      25,
        'lenguajes': ['Python', 'R'],
        'nivel':     'junior',
        'activo':    False
    }
}

# Acceso anidado: d['clave_externa']['clave_interna']
print(usuarios['ana_g']['nombre'])
print(usuarios['carlos_m']['lenguajes'][0])   # primer lenguaje

# Acceso seguro con .get() encadenado
print(usuarios.get('x_user', {}).get('nombre', 'Sin nombre'))  # sin KeyError

In [ ]:
# Iterar y procesar diccionarios anidados
print("=" * 50)
print(f"{'DIRECTORIO DE DESARROLLADORES':^50}")
print("=" * 50)

for username, info in usuarios.items():
    estado = "✅ Activo" if info['activo'] else "⏸️  Inactivo"
    langs  = ', '.join(info['lenguajes'])
    print(f"\n👤 @{username} ({estado})")
    print(f"   Nombre:    {info['nombre']}")
    print(f"   Nivel:     {info['nivel'].title()}")
    print(f"   Edad:      {info['edad']} años")
    print(f"   Lenguajes: {langs}")

# Filtrar usuarios activos con Python en su stack
print("\n--- Usuarios activos con Python ---")
python_users = [
    info['nombre']
    for info in usuarios.values()
    if info['activo'] and 'Python' in info['lenguajes']
]
for nombre in python_users:
    print(f"  • {nombre}")

### 6.6.2 Modificar diccionarios anidados

In [ ]:
# Agregar nuevo usuario
usuarios['diego_r'] = {
    'nombre':    'Diego Ramírez',
    'edad':      22,
    'lenguajes': ['JavaScript', 'TypeScript'],
    'nivel':     'junior',
    'activo':    True
}

# Modificar campo de usuario existente
usuarios['bea_l']['activo'] = True
usuarios['bea_l']['nivel']  = 'mid'

# Agregar un lenguaje a la lista de un usuario
usuarios['ana_g']['lenguajes'].append('Spark')
print(f"Lenguajes de Ana: {usuarios['ana_g']['lenguajes']}")

# Agregar campo nuevo a todos los usuarios
for username, info in usuarios.items():
    info['proyectos'] = 0    # inicializar contador

print(f"\nTotal usuarios: {len(usuarios)}")
print(f"Campos por usuario: {list(usuarios['ana_g'].keys())}")

### 6.6.3 Lista de diccionarios — datos tabulares

In [ ]:
# Lista de diccionarios — equivalente a un data.frame de R (sin pandas)
# Cada diccionario = una fila; las claves = columnas

productos = [
    {'nombre': 'Laptop',    'precio': 15999, 'stock': 10, 'categoria': 'electrónica'},
    {'nombre': 'Mouse',     'precio':   349, 'stock': 50, 'categoria': 'electrónica'},
    {'nombre': 'Teclado',   'precio':   799, 'stock': 30, 'categoria': 'electrónica'},
    {'nombre': 'Silla',     'precio':  2500, 'stock': 15, 'categoria': 'muebles'},
    {'nombre': 'Monitor',   'precio':  5999, 'stock':  8, 'categoria': 'electrónica'},
    {'nombre': 'Escritorio','precio':  4500, 'stock':  5, 'categoria': 'muebles'},
]

# Acceso: lista[índice]['clave']
print(f"Primer producto: {productos[0]['nombre']}")
print(f"Precio del monitor: ${productos[4]['precio']:,}")

# Filtrar — equivalente a dplyr::filter()
electronica = [p for p in productos if p['categoria'] == 'electrónica']
print(f"\nElectrónica: {len(electronica)} productos")

# Ordenar — equivalente a dplyr::arrange()
por_precio = sorted(productos, key=lambda p: p['precio'], reverse=True)
print("\nPor precio (mayor a menor):")
for p in por_precio:
    print(f"  {p['nombre']:<12}: ${p['precio']:>7,} (stock: {p['stock']})")

# Agregar campo calculado — equivalente a dplyr::mutate()
for p in productos:
    p['valor_inventario'] = p['precio'] * p['stock']

total = sum(p['valor_inventario'] for p in productos)
print(f"\nValor total del inventario: ${total:,}")

::: {.callout-note title="R vs Python — Datos tabulares"}
```r
# En R — data.frame o tibble
productos <- data.frame(
  nombre    = c('Laptop', 'Mouse', 'Monitor'),
  precio    = c(15999, 349, 5999),
  stock     = c(10, 50, 8),
  categoria = c('electrónica', 'electrónica', 'electrónica')
)

# Operaciones con dplyr
library(dplyr)
productos %>% filter(categoria == 'electrónica') %>% arrange(desc(precio))
productos %>% mutate(valor_inv = precio * stock)
sum(productos$precio * productos$stock)
```
```python
# En Python puro — lista de diccionarios
productos = [
    {'nombre': 'Laptop', 'precio': 15999, 'stock': 10},
    {'nombre': 'Mouse',  'precio': 349,   'stock': 50},
]

# Filtrar
[p for p in productos if p['precio'] > 1000]

# Ordenar
sorted(productos, key=lambda p: p['precio'], reverse=True)

# Agregar campo calculado
for p in productos: p['valor_inv'] = p['precio'] * p['stock']

# ← En la práctica real, se usa pandas (capítulo posterior)
import pandas as pd
df = pd.DataFrame(productos)   # convierte la lista en DataFrame
```
La lista de diccionarios es la forma de trabajar con datos tabulares en Python puro. Cuando el dataset sea más grande o complejo, se usa **pandas** — que convierte estas listas en DataFrames similares a R.
:::

---

## 6.7 Patrones Avanzados

### 6.7.1 `defaultdict` — valores por defecto automáticos

In [ ]:
from collections import defaultdict

# defaultdict crea automáticamente el valor por defecto
# cuando se accede a una clave que no existe

# Agrupar elementos por categoría — patrón muy común
productos_lista = [
    ('Laptop', 'electrónica'),
    ('Mouse', 'electrónica'),
    ('Silla', 'muebles'),
    ('Monitor', 'electrónica'),
    ('Escritorio', 'muebles'),
    ('Audífonos', 'electrónica'),
]

# Sin defaultdict — más verboso
grupos_manual = {}
for producto, categoria in productos_lista:
    if categoria not in grupos_manual:
        grupos_manual[categoria] = []   # inicializar si no existe
    grupos_manual[categoria].append(producto)

# Con defaultdict — más limpio
grupos = defaultdict(list)   # list es el factory de valores por defecto
for producto, categoria in productos_lista:
    grupos[categoria].append(producto)   # no necesitas verificar si existe

print("Grupos con defaultdict:")
for cat, prods in sorted(grupos.items()):
    print(f"  {cat}: {prods}")

# defaultdict(int) — contador
palabras = ['python', 'r', 'python', 'julia', 'python', 'r', 'sql']
contador = defaultdict(int)
for p in palabras:
    contador[p] += 1   # int() → 0 por defecto, nunca KeyError

print(f"\nContador: {dict(sorted(contador.items()))}")

### 6.7.2 `Counter` — contador especializado

In [ ]:
from collections import Counter

# Counter es un dict especializado en contar frecuencias
# Equivale a table() de R

lenguajes_equipo = [
    'Python', 'R', 'Python', 'SQL', 'Python',
    'R', 'Julia', 'Python', 'SQL', 'R'
]

contador = Counter(lenguajes_equipo)
print(f"Counter: {contador}")
print(f"Más común: {contador.most_common(3)}")  # top 3
print(f"Usos de Python: {contador['Python']}")
print(f"Usos de Go: {contador['Go']}")           # 0, no KeyError

# Counter con strings — cuenta caracteres
texto = "mississippi"
caracteres = Counter(texto)
print(f"\nCaracteres en 'mississippi': {dict(sorted(caracteres.items()))}")
print(f"Letra más común: {caracteres.most_common(1)}")

# Operaciones aritméticas entre Counters
equipo_a = Counter(['Python', 'R', 'Python', 'SQL'])
equipo_b = Counter(['Python', 'Julia', 'Python', 'R', 'R'])
print(f"\nEquipo A: {dict(equipo_a)}")
print(f"Equipo B: {dict(equipo_b)}")
print(f"Unión:    {dict(equipo_a + equipo_b)}")

::: {.callout-note title="R vs Python — Contar frecuencias"}
```r
# En R — table() es el equivalente directo
lenguajes <- c('Python', 'R', 'Python', 'SQL', 'Python', 'R')
table(lenguajes)          # tabla de frecuencias
sort(table(lenguajes), decreasing=TRUE)  # ordenado
names(sort(table(lenguajes), decreasing=TRUE))[1]  # más común
```
```python
# En Python — Counter del módulo collections
from collections import Counter
lenguajes = ['Python', 'R', 'Python', 'SQL', 'Python', 'R']

contador = Counter(lenguajes)
contador.most_common()        # ordenado por frecuencia
contador.most_common(1)[0][0] # el más común
```
Ambos son muy similares en funcionalidad. `Counter` tiene la ventaja de no lanzar error para claves no vistas (retorna 0).
:::

### 6.7.3 Combinar diccionarios

In [ ]:
config_base  = {'tema': 'claro', 'idioma': 'español', 'volumen': 70, 'zoom': 1.0}
config_user  = {'tema': 'oscuro', 'volumen': 85, 'fuente': 14}

# Forma 1: .update() — modifica en lugar
combinado1 = config_base.copy()    # copiar para no modificar el original
combinado1.update(config_user)     # config_user sobrescribe donde hay conflicto
print(f"update():       {combinado1}")

# Forma 2: operador | (Python 3.9+) — retorna nuevo diccionario
combinado2 = config_base | config_user    # el derecho sobrescribe al izquierdo
print(f"| (3.9+):       {combinado2}")

# Forma 3: ** (desempaquetado) — compatible con Python 3.5+
combinado3 = {**config_base, **config_user}
print(f"** (3.5+):      {combinado3}")

# Los tres producen el mismo resultado
print(f"\n¿Iguales? {combinado1 == combinado2 == combinado3}")

---

## 6.8 Errores Comunes

In [ ]:
# Error 1: KeyError al acceder a clave inexistente
d = {'a': 1, 'b': 2}

try:
    print(d['c'])     # ❌ KeyError
except KeyError as e:
    print(f"KeyError: {e}")

# ✅ Solución: usar .get()
print(d.get('c', 'no existe'))   # seguro

# ✅ O verificar antes
if 'c' in d:
    print(d['c'])
else:
    print("Clave 'c' no existe")

In [ ]:
# Error 2: Modificar el diccionario mientras se itera (¡muy común!)
d = {'a': 1, 'b': 2, 'c': 3, 'd': 4}

# ❌ RuntimeError: dictionary changed size during iteration
# for k in d:
#     if d[k] < 3:
#         del d[k]

# ✅ Solución 1: iterar sobre una copia de las claves
for k in list(d.keys()):      # list() crea una copia
    if d[k] < 3:
        del d[k]
print(f"Tras eliminar menores a 3: {d}")

# ✅ Solución 2: dict comprehension (más pythónica)
d = {'a': 1, 'b': 2, 'c': 3, 'd': 4}
d = {k: v for k, v in d.items() if v >= 3}
print(f"Con comprehension: {d}")

In [ ]:
# Error 3: Confundir 'in' — verifica CLAVES, no valores
d = {'nombre': 'Ana', 'edad': 28}

print('nombre' in d)    # True  — verifica claves ✅
print('Ana' in d)       # False — 'Ana' es un VALOR, no una clave ⚠️
print('Ana' in d.values())  # True — correcto para buscar valores

# Error 4: Usar lista como clave (las listas son mutables = no hashable)
try:
    d_malo = {[1, 2]: 'valor'}   # ❌ TypeError
except TypeError as e:
    print(f"\nTypeError: {e}")

# ✅ Usar tupla en su lugar
d_bueno = {(1, 2): 'valor'}     # ✅ tuplas son inmutables
print(f"Clave tupla: {d_bueno[(1, 2)]}")

In [ ]:
# Error 5: Copiar diccionario con = (alias, no copia)
original = {'a': 1, 'b': [1, 2, 3]}
alias    = original          # ¡Mismo objeto!

alias['a'] = 99
print(f"original: {original}")   # También cambió

# ✅ Copia superficial con .copy()
original = {'a': 1, 'b': [1, 2, 3]}
copia = original.copy()

copia['a'] = 99
print(f"\noriginal: {original}")   # 'a' no cambió

# ⚠️ PERO los valores mutables (listas, dicts) sí se comparten
copia['b'].append(4)
print(f"original['b']: {original['b']}")  # ¡También cambió!

# ✅ Para copia profunda, usar copy.deepcopy()
import copy
original = {'a': 1, 'b': [1, 2, 3]}
copia_profunda = copy.deepcopy(original)
copia_profunda['b'].append(4)
print(f"\noriginal['b'] (deepcopy): {original['b']}")  # No cambió

::: {.callout-warning title="🚨 Tabla de Errores Clásicos — Diccionarios"}
| Error | Código Erróneo | Código Correcto |
|---|---|---|
| Clave inexistente | `d['no_existe']` | `d.get('no_existe', default)` |
| Modificar al iterar | `for k in d: del d[k]` | `for k in list(d.keys()): ...` |
| Buscar valor con `in` | `'valor' in d` | `'valor' in d.values()` |
| Lista como clave | `{[1,2]: 'x'}` | `{(1,2): 'x'}` (tupla) |
| Copia superficial | `copia = d` | `copia = d.copy()` |
| Copia de anidados | `d.copy()` | `copy.deepcopy(d)` |
| sort() en dict | No existe | `sorted(d.items(), key=...)` |
:::

---

## 🧩 Buenas Prácticas

In [ ]:
# 1. Prefer .get() sobre acceso directo cuando la clave puede no existir
config = {'tema': 'oscuro'}

# ❌ Frágil
try:
    print(config['volumen'])
except KeyError:
    volumen = 70

# ✅ Limpio
volumen = config.get('volumen', 70)
print(f"Volumen: {volumen}")

# 2. Usar dict comprehension para transformar diccionarios
precios = {'a': 100, 'b': 250, 'c': 80}

# ❌ Verboso
con_iva = {}
for k, v in precios.items():
    con_iva[k] = round(v * 1.16, 2)

# ✅ Conciso
con_iva = {k: round(v * 1.16, 2) for k, v in precios.items()}
print(f"\nCon IVA: {con_iva}")

# 3. Usar Counter para frecuencias en lugar de loop manual
from collections import Counter
palabras = ['python', 'r', 'python', 'sql', 'python', 'r']

# ❌ Loop manual
conteo = {}
for p in palabras:
    conteo[p] = conteo.get(p, 0) + 1

# ✅ Counter
conteo = Counter(palabras)
print(f"\nConteo: {dict(conteo)}")

# 4. Desempaquetar pares al iterar
personas = {'Ana': 28, 'Carlos': 32, 'Sofía': 25}

# ❌ Acceso manual
for nombre in personas:
    edad = personas[nombre]
    print(f"{nombre}: {edad}")

# ✅ Desempaquetar con .items()
for nombre, edad in personas.items():
    print(f"{nombre}: {edad}")

---

## ✏️ Ejercicios Prácticos

---

### 🟢 Nivel 1 — Fundamentos

#### Ejercicio 1.1: Crear y Consultar Diccionarios

In [ ]:
# Ejercicio 1.1
# Crea un diccionario que represente tu perfil como analista de datos:
# campos: nombre, edad, ciudad, herramientas (lista), experiencia_años, lenguaje_fav
#
# Luego:
# a) Imprime cada campo con formato: "Campo: valor"
# b) Imprime solo las herramientas, numeradas desde 1
# c) Usa .get() para intentar acceder a un campo 'salario' (no existe)
#    y muestra un mensaje amigable si no está
# d) Agrega el campo 'certificaciones' con una lista vacía,
#    y luego agrega dos certificaciones a esa lista

# Tu código aquí:
perfil = {}


::: {.callout-tip title="✅ Solución Ejercicio 1.1" collapse="true"}
```python
perfil = {
    'nombre':          'Ana García',
    'edad':            28,
    'ciudad':          'CDMX',
    'herramientas':    ['Python', 'R', 'SQL', 'Tableau'],
    'experiencia_años': 4,
    'lenguaje_fav':    'Python'
}

# a)
for campo, valor in perfil.items():
    print(f"{campo.replace('_', ' ').title()}: {valor}")

# b)
print("\nHerramientas:")
for i, herr in enumerate(perfil['herramientas'], 1):
    print(f"  {i}. {herr}")

# c)
salario = perfil.get('salario', 'Información no disponible')
print(f"\nSalario: {salario}")

# d)
perfil['certificaciones'] = []
perfil['certificaciones'].append('AWS Cloud Practitioner')
perfil['certificaciones'].append('Google Data Analytics')
print(f"\nCertificaciones: {perfil['certificaciones']}")
```
:::

#### Ejercicio 1.2: Diccionario desde Listas

In [ ]:
# Ejercicio 1.2
# Tienes estas dos listas paralelas de datos.
# Construye diccionarios y realiza consultas.

productos = ['Laptop', 'Mouse', 'Teclado', 'Monitor', 'Audífonos', 'Webcam']
precios   = [15999,    349,     799,        5999,       1299,        899]
stocks    = [10,       50,      30,         8,          20,          12]

# a) Crea un dict {producto: precio} usando dict comprehension + zip()

# b) Crea un dict {producto: (precio, stock)} — valor es una tupla

# c) Imprime el producto más caro y el más barato

# d) Imprime todos los productos con stock menor a 15

# Tu código aquí:


::: {.callout-tip title="✅ Solución Ejercicio 1.2" collapse="true"}
```python
productos = ['Laptop', 'Mouse', 'Teclado', 'Monitor', 'Audífonos', 'Webcam']
precios   = [15999, 349, 799, 5999, 1299, 899]
stocks    = [10, 50, 30, 8, 20, 12]

# a)
precios_dict = {p: pr for p, pr in zip(productos, precios)}
print(f"a) {precios_dict}")

# b)
catalogo = {p: (pr, st) for p, pr, st in zip(productos, precios, stocks)}
print(f"\nb) {catalogo}")

# c)
mas_caro  = max(precios_dict, key=precios_dict.get)
mas_barat = min(precios_dict, key=precios_dict.get)
print(f"\nc) Más caro:   {mas_caro} (${precios_dict[mas_caro]:,})")
print(f"   Más barato: {mas_barat} (${precios_dict[mas_barat]:,})")

# d)
print("\nd) Bajo stock (< 15):")
for prod, (precio, stock) in catalogo.items():
    if stock < 15:
        print(f"   {prod}: {stock} unidades")
```
:::

#### Ejercicio 1.3: Manipular y Recorrer

In [ ]:
# Ejercicio 1.3
# Dada esta configuración de aplicación, realiza las operaciones indicadas.

config = {
    'tema':         'claro',
    'idioma':       'español',
    'volumen':      70,
    'notif':        True,
    'fuente_size':  12,
    'zoom':         1.0
}

# a) Muestra todos los pares con formato "  clave → valor"

# b) Cambia el tema a 'oscuro' y el volumen a 85

# c) Agrega 'brillo': 80 solo si no existe ya en el config
#    (usa setdefault)

# d) Elimina 'notif' y guarda su valor en una variable

# e) Muestra el diccionario final ordenado por clave (alfabéticamente)

# Tu código aquí:


::: {.callout-tip title="✅ Solución Ejercicio 1.3" collapse="true"}
```python
config = {
    'tema': 'claro', 'idioma': 'español', 'volumen': 70,
    'notif': True, 'fuente_size': 12, 'zoom': 1.0
}

# a)
for k, v in config.items():
    print(f"  {k} → {v}")

# b)
config['tema']   = 'oscuro'
config['volumen'] = 85

# c)
config.setdefault('brillo', 80)

# d)
notif = config.pop('notif')
print(f"\nd) notif eliminado: {notif}")

# e)
print("\ne) Config final (ordenado):")
for k, v in sorted(config.items()):
    print(f"  {k}: {v}")
```
:::

---

### 🟡 Nivel 2 — Aplicación

#### Ejercicio 2.1: Contador de Palabras

In [ ]:
# Ejercicio 2.1
# Analiza la frecuencia de palabras en un texto.
# (Similar a table(strsplit(texto, ' ')[[1]]) en R)

texto = (
    "Python es un lenguaje de programación poderoso y fácil de aprender. "
    "Python es versátil y se usa en web, ciencia de datos e inteligencia artificial. "
    "Aprender Python es una excelente inversión de tiempo. "
    "La ciencia de datos usa Python y R como lenguajes principales."
)

# a) Cuenta la frecuencia de cada palabra (ignora puntuación, minúsculas)
#    Pista: texto.lower().split() y str.strip('.,!?;:')

# b) Muestra el top 10 de palabras más frecuentes
#    con una barra de progreso de █ proporcional a su frecuencia

# c) ¿Cuántas palabras ÚNICAS hay? ¿Cuántas palabras en total?

# d) Filtra las palabras con frecuencia > 2 y muéstralas

# Tu código aquí:


::: {.callout-tip title="✅ Solución Ejercicio 2.1" collapse="true"}
```python
from collections import Counter

texto = (
    "Python es un lenguaje de programación poderoso y fácil de aprender. "
    "Python es versátil y se usa en web, ciencia de datos e inteligencia artificial. "
    "Aprender Python es una excelente inversión de tiempo. "
    "La ciencia de datos usa Python y R como lenguajes principales."
)

# a) Contar frecuencias
palabras = [p.strip('.,!?;:') for p in texto.lower().split()]
freq = Counter(palabras)

# b) Top 10 con barra
print("TOP 10 PALABRAS")
print("-" * 35)
max_freq = freq.most_common(1)[0][1]
for palabra, count in freq.most_common(10):
    barra = '█' * int(count / max_freq * 20)
    print(f"  {palabra:<15} {count:>2}  {barra}")

# c) Estadísticas
print(f"\nc) Palabras únicas: {len(freq)}")
print(f"   Total palabras:  {sum(freq.values())}")

# d) Frecuencia > 2
frec_alta = {p: c for p, c in freq.items() if c > 2}
print(f"\nd) Con frecuencia > 2: {dict(sorted(frec_alta.items(), key=lambda x: -x[1]))}")
```
:::

#### Ejercicio 2.2: Dict Comprehensions Avanzadas

In [ ]:
# Ejercicio 2.2
# Resuelve cada punto con una dict comprehension en una sola línea.
# (En R usarías setNames + sapply o vapply)

alumnos = {
    'Sofía':    [90, 85, 95, 88],
    'Miguel':   [72, 68, 74, 70],
    'Valeria':  [95, 98, 93, 97],
    'Diego':    [61, 58, 65, 63],
    'Camila':   [84, 87, 80, 83],
}

# a) Crea un dict {nombre: promedio} (redondeado a 1 decimal)

# b) Crea un dict {nombre: 'Aprobado'/'Reprobado'}
#    (aprueba si promedio >= 70)

# c) Crea un dict {nombre: rango} donde rango = max - min de sus calificaciones

# d) Filtra solo los alumnos con promedio >= 85 y sus promedios

# e) Invierte el dict del punto (a): {promedio: nombre}
#    (pista: usa round() para evitar problemas de float)

# Tu código aquí:


::: {.callout-tip title="✅ Solución Ejercicio 2.2" collapse="true"}
```python
alumnos = {
    'Sofía':   [90, 85, 95, 88], 'Miguel':  [72, 68, 74, 70],
    'Valeria': [95, 98, 93, 97], 'Diego':   [61, 58, 65, 63],
    'Camila':  [84, 87, 80, 83],
}

# a)
promedios = {n: round(sum(c)/len(c), 1) for n, c in alumnos.items()}
print(f"a) {promedios}")

# b)
estados = {n: 'Aprobado' if p >= 70 else 'Reprobado' for n, p in promedios.items()}
print(f"b) {estados}")

# c)
rangos = {n: max(c) - min(c) for n, c in alumnos.items()}
print(f"c) {rangos}")

# d)
sobresalientes = {n: p for n, p in promedios.items() if p >= 85}
print(f"d) {sobresalientes}")

# e)
invertido = {p: n for n, p in promedios.items()}
print(f"e) {invertido}")
```
:::

#### Ejercicio 2.3: Agrupar y Analizar

In [ ]:
# Ejercicio 2.3
# Dada esta lista de ventas, agrupa y analiza por categoría.
# (Equivalente a dplyr::group_by() + summarise() en R)

ventas = [
    {'producto': 'Laptop',     'categoria': 'electrónica', 'monto': 15999, 'mes': 'Ene'},
    {'producto': 'Silla',      'categoria': 'muebles',     'monto':  2500, 'mes': 'Ene'},
    {'producto': 'Mouse',      'categoria': 'electrónica', 'monto':   349, 'mes': 'Feb'},
    {'producto': 'Escritorio', 'categoria': 'muebles',     'monto':  4500, 'mes': 'Feb'},
    {'producto': 'Monitor',    'categoria': 'electrónica', 'monto':  5999, 'mes': 'Mar'},
    {'producto': 'Audífonos',  'categoria': 'electrónica', 'monto':  1299, 'mes': 'Mar'},
    {'producto': 'Lámpara',    'categoria': 'muebles',     'monto':   850, 'mes': 'Mar'},
    {'producto': 'Teclado',    'categoria': 'electrónica', 'monto':   799, 'mes': 'Ene'},
]

# a) Agrupa los montos por categoría: {categoria: [lista de montos]}
#    Usa defaultdict

# b) Calcula para cada categoría: total, promedio, máximo, número de ventas

# c) Agrupa por mes: {mes: total_vendido}

# d) ¿Cuál fue el mes con mayor ventas?

# Tu código aquí:


::: {.callout-tip title="✅ Solución Ejercicio 2.3" collapse="true"}
```python
from collections import defaultdict

ventas = [
    {'producto': 'Laptop',    'categoria': 'electrónica', 'monto': 15999, 'mes': 'Ene'},
    {'producto': 'Silla',     'categoria': 'muebles',     'monto':  2500, 'mes': 'Ene'},
    {'producto': 'Mouse',     'categoria': 'electrónica', 'monto':   349, 'mes': 'Feb'},
    {'producto': 'Escritorio','categoria': 'muebles',     'monto':  4500, 'mes': 'Feb'},
    {'producto': 'Monitor',   'categoria': 'electrónica', 'monto':  5999, 'mes': 'Mar'},
    {'producto': 'Audífonos', 'categoria': 'electrónica', 'monto':  1299, 'mes': 'Mar'},
    {'producto': 'Lámpara',   'categoria': 'muebles',     'monto':   850, 'mes': 'Mar'},
    {'producto': 'Teclado',   'categoria': 'electrónica', 'monto':   799, 'mes': 'Ene'},
]

# a) Agrupar montos por categoría
montos_cat = defaultdict(list)
for v in ventas:
    montos_cat[v['categoria']].append(v['monto'])
print("a)", dict(montos_cat))

# b) Resumen por categoría
print("\nb) Resumen por categoría:")
for cat, montos in montos_cat.items():
    print(f"   {cat}:")
    print(f"     Total:    ${sum(montos):,}")
    print(f"     Promedio: ${sum(montos)/len(montos):,.0f}")
    print(f"     Máximo:   ${max(montos):,}")
    print(f"     Ventas:   {len(montos)}")

# c) Total por mes
por_mes = defaultdict(int)
for v in ventas:
    por_mes[v['mes']] += v['monto']
print(f"\nc) Por mes: {dict(por_mes)}")

# d) Mejor mes
mejor_mes = max(por_mes, key=por_mes.get)
print(f"d) Mejor mes: {mejor_mes} (${por_mes[mejor_mes]:,})")
```
:::

---

### 🔴 Nivel 3 — Integración

#### Ejercicio 3.1: Sistema de Calificaciones Completo

In [ ]:
# Ejercicio 3.1
# Construye un sistema completo de análisis académico usando
# diccionarios anidados. Equivale a un análisis de data.frame
# complejo en R.

# Estructura: {alumno: {materia: [calificaciones]}}
datos = {
    'Sofía':   {'Matemáticas': [90, 88, 92], 'Estadística': [95, 91, 93], 'Programación': [97, 95, 98]},
    'Miguel':  {'Matemáticas': [72, 68, 75], 'Estadística': [70, 74, 71], 'Programación': [68, 72, 70]},
    'Valeria': {'Matemáticas': [85, 88, 90], 'Estadística': [92, 89, 94], 'Programación': [88, 91, 87]},
    'Diego':   {'Matemáticas': [60, 63, 65], 'Estadística': [58, 62, 61], 'Programación': [65, 68, 64]},
}

# a) Calcula el promedio por alumno por materia:
#    resultado: {alumno: {materia: promedio}}

# b) Calcula el promedio general por alumno (promedio de sus materias)

# c) Calcula el promedio por materia (promedio de todos los alumnos)

# d) Imprime un reporte tabular:
#    Alumno | Matemáticas | Estadística | Programación | General

# e) ¿Qué alumno tiene el mayor promedio? ¿En qué materia?

# Tu código aquí:


::: {.callout-tip title="✅ Solución Ejercicio 3.1" collapse="true"}
```python
datos = {
    'Sofía':   {'Matemáticas': [90,88,92], 'Estadística': [95,91,93], 'Programación': [97,95,98]},
    'Miguel':  {'Matemáticas': [72,68,75], 'Estadística': [70,74,71], 'Programación': [68,72,70]},
    'Valeria': {'Matemáticas': [85,88,90], 'Estadística': [92,89,94], 'Programación': [88,91,87]},
    'Diego':   {'Matemáticas': [60,63,65], 'Estadística': [58,62,61], 'Programación': [65,68,64]},
}
materias = ['Matemáticas', 'Estadística', 'Programación']

# a) Promedios por alumno por materia
promedios = {
    alumno: {mat: round(sum(califs)/len(califs), 1)
             for mat, califs in materias_d.items()}
    for alumno, materias_d in datos.items()
}

# b) Promedio general por alumno
general = {a: round(sum(m.values())/len(m), 1) for a, m in promedios.items()}

# c) Promedio por materia
por_materia = {
    mat: round(sum(promedios[a][mat] for a in promedios) / len(promedios), 1)
    for mat in materias
}

# d) Reporte tabular
print(f"{'Alumno':<12}" + "".join(f"{m:>15}" for m in materias) + f"{'General':>10}")
print("-" * (12 + 15*len(materias) + 10))
for alumno in promedios:
    fila = f"{alumno:<12}"
    for mat in materias:
        fila += f"{promedios[alumno][mat]:>15}"
    fila += f"{general[alumno]:>10}"
    print(fila)
print("-" * (12 + 15*len(materias) + 10))
fila_prom = f"{'Promedio':<12}"
for mat in materias:
    fila_prom += f"{por_materia[mat]:>15}"
print(fila_prom)

# e) Mejor alumno y materia
mejor_alumno = max(general, key=general.get)
mejor_materia = max(por_materia, key=por_materia.get)
print(f"\nMejor alumno:  {mejor_alumno} ({general[mejor_alumno]})")
print(f"Mejor materia: {mejor_materia} ({por_materia[mejor_materia]})")
```
:::

#### Ejercicio 3.2: Caché de Fibonacci con Memoización

In [ ]:
# Ejercicio 3.2
# Implementa la sucesión de Fibonacci usando un diccionario como caché
# (memoización). Esto evita recalcular valores ya computados.
#
# La memoización es un patrón fundamental en programación — es equivalente
# a la técnica de 'memoize' del paquete memoise de R.
#
# Sin caché: fib(40) requiere ~2^40 llamadas recursivas
# Con caché:  fib(40) requiere solo 40 cálculos
#
# a) Implementa fibonacci_cache(n, cache={}) que:
#    - Si n está en cache, lo retorna directamente
#    - Si no, lo calcula y lo guarda en cache antes de retornar
#    - Casos base: fib(0)=0, fib(1)=1
#
# b) Calcula fib(0) a fib(20) e imprime los resultados
#
# c) Imprime el contenido del caché al final
#    ¿Cuántas entradas tiene?

# Tu código aquí:


::: {.callout-tip title="✅ Solución Ejercicio 3.2" collapse="true"}
```python
def fibonacci_cache(n, cache={}):
    """Calcula el n-ésimo número de Fibonacci con memoización."""
    if n in cache:
        return cache[n]
    if n <= 1:
        return n
    resultado = fibonacci_cache(n-1, cache) + fibonacci_cache(n-2, cache)
    cache[n] = resultado
    return resultado

# b) Calcular fib(0) a fib(20)
print("Sucesión de Fibonacci (con caché):")
for i in range(21):
    print(f"  fib({i:2d}) = {fibonacci_cache(i):,}")

# c) Contenido del caché
cache_local = {}
for i in range(21):
    fibonacci_cache(i, cache_local)
print(f"\nEntradas en caché: {len(cache_local)}")
print(f"Cache (primeras 5): {dict(list(cache_local.items())[:5])}")
```
:::

---

## 🚀 Mini Proyecto: Traductor de Código Morse Bidireccional

In [ ]:
# Mini Proyecto: Sistema completo de traducción Morse
# Demuestra diccionarios normales, invertidos, y dict comprehensions

MORSE = {
    'A': '.-',   'B': '-...',  'C': '-.-.',  'D': '-..',   'E': '.',
    'F': '..-.',  'G': '--.',   'H': '....',  'I': '..',    'J': '.---',
    'K': '-.-',   'L': '.-..',  'M': '--',    'N': '-.',    'O': '---',
    'P': '.--.',  'Q': '--.-',  'R': '.-.',   'S': '...',   'T': '-',
    'U': '..-',   'V': '...-',  'W': '.--',   'X': '-..-',  'Y': '-.--',
    'Z': '--..',  '0': '-----', '1': '.----', '2': '..---', '3': '...--',
    '4': '....-', '5': '.....', '6': '-....', '7': '--...', '8': '---..',
    '9': '----.',  ' ': '/'
}

# Invertir diccionario con dict comprehension — O(n) una sola vez
MORSE_INV = {v: k for k, v in MORSE.items()}

def codificar(texto):
    """Convierte texto a código Morse."""
    resultado = []
    for char in texto.upper():
        codigo = MORSE.get(char)
        resultado.append(codigo if codigo else '?')
    return ' '.join(resultado)

def decodificar(morse):
    """Convierte código Morse a texto."""
    resultado = []
    for bloque in morse.split(' / '):
        palabra = ''.join(MORSE_INV.get(s, '?') for s in bloque.split())
        resultado.append(palabra)
    return ' '.join(resultado)

def analizar_morse(texto):
    """Retorna estadísticas del texto en Morse."""
    morse = codificar(texto)
    simbolos = [s for s in morse.split() if s != '/']
    puntos   = sum(s.count('.') for s in simbolos)
    rayas    = sum(s.count('-') for s in simbolos)
    return {
        'texto_original':  texto,
        'morse':           morse,
        'caracteres':      len(texto.replace(' ', '')),
        'señales_total':   len(simbolos),
        'puntos':          puntos,
        'rayas':           rayas,
        'relacion_p_r':   round(puntos / rayas, 2) if rayas else None
    }

# Demostración
mensajes = ["SOS", "HOLA PYTHON", "MORSE CODE"]

print("=" * 60)
print(f"{'TRADUCTOR DE CÓDIGO MORSE':^60}")
print("=" * 60)

for msg in mensajes:
    stats = analizar_morse(msg)
    codif = stats['morse']
    decodif = decodificar(codif)

    print(f"\n📡 Original:   {stats['texto_original']}")
    print(f"   Morse:      {codif}")
    print(f"   Decodif.:   {decodif}")
    print(f"   Chars:      {stats['caracteres']} | "
          f"Señales: {stats['señales_total']} | "
          f"Puntos: {stats['puntos']} | "
          f"Rayas: {stats['rayas']}")

# Estadísticas del diccionario Morse
print("\n" + "=" * 60)
print("ESTADÍSTICAS DEL CÓDIGO MORSE")
print("-" * 60)
long_senal = {k: len(v) for k, v in MORSE.items() if v != '/'}
mas_corto = min(long_senal, key=long_senal.get)
mas_largo = max(long_senal, key=long_senal.get)
print(f"Caracteres definidos: {len(MORSE)}")
print(f"Señal más corta:  '{mas_corto}' → {MORSE[mas_corto]}")
print(f"Señal más larga:  '{mas_largo}' → {MORSE[mas_largo]}")

# Distribución de longitudes
from collections import Counter
dist = Counter(len(v) for v in MORSE.values() if v != '/')
print("Distribución de longitudes:")
for long, count in sorted(dist.items()):
    print(f"  {long} símbolos: {count} caracteres")

---

## 🏆 Desafíos

::: {.callout-warning title="Desafío 1 — Histograma de Letras"}
Dado cualquier texto, construye un histograma horizontal de frecuencia de letras
(solo letras, sin espacios ni puntuación, sin distinción de mayúsculas).
Ordénalo de mayor a menor frecuencia. Usa `Counter` y barras `█` proporcionales.
Compara el resultado con la frecuencia esperada del español: e, a, o, s, n, r, i...
:::

::: {.callout-warning title="Desafío 2 — Índice invertido"}
Dado un diccionario `{documento: [palabras]}`, construye el índice invertido:
`{palabra: [documentos_donde_aparece]}`. Este es el principio de un motor de búsqueda.
```python
docs = {
    'doc1': ['python', 'datos', 'análisis'],
    'doc2': ['r', 'estadística', 'datos'],
    'doc3': ['python', 'machine', 'learning', 'datos']
}
# Resultado esperado:
# {'python': ['doc1', 'doc3'], 'datos': ['doc1', 'doc2', 'doc3'], ...}
```
:::

::: {.callout-warning title="Desafío 3 — Group by propio"}
Implementa una función `group_by(lista_dicts, campo)` que agrupe una lista de
diccionarios por el valor de un campo, devolviendo `{valor: [lista de registros]}`.
Equivalente a `dplyr::group_by()` de R. Pruébala con las ventas del Ejercicio 2.3.
:::

::: {.callout-warning title="Desafío 4 — Comparación de estilos"}
En R, `table(v)` genera una tabla de frecuencias en una sola llamada.
En Python, `Counter(v)` hace lo mismo. Pero `Counter` tiene muchas más
capacidades (aritmética, most_common, etc.).
Escribe un análisis comparativo: dado el mismo dataset, resuelve el mismo
problema de 3 formas: (1) loop manual con dict, (2) defaultdict, (3) Counter.
Comenta cuándo usarías cada uno.
:::

---

## 📝 Autoevaluación

| Pregunta | Tu respuesta |
|---|---|
| ¿Cuál es la diferencia entre `d['k']` y `d.get('k')`? | |
| ¿Cómo recorres claves y valores a la vez? | |
| ¿Qué hace `{v: k for k, v in d.items()}`? | |
| ¿Cuándo usarías `defaultdict` en lugar de `dict`? | |
| ¿Por qué no se puede usar una lista como clave de dict? | |
| ¿Cómo combinas dos dicts en Python 3.9+? ¿Y en 3.5+? | |
| ¿Qué diferencia hay entre `.pop()` y `del`? | |
| ¿Cómo buscas un valor (no clave) en un diccionario? | |
| Equivalente en R de `d.get('k', default)` | |
| ¿Qué hace `Counter.most_common(3)`? Equivalente en R. | |

---

## 🗒️ Resumen del Capítulo

| Operación | R (`list` nombrada) | Python (`dict`) |
|---|---|---|
| Crear | `list(a=1, b=2)` | `{'a': 1, 'b': 2}` |
| Acceder | `l$clave` o `l[['clave']]` | `d['clave']` |
| Acceso seguro | `l$clave` (NULL si no existe) | `d.get('clave', default)` |
| Agregar/modificar | `l$nueva <- v` | `d['nueva'] = v` |
| Eliminar | `l$clave <- NULL` | `del d['clave']` |
| Eliminar + obtener | No directo | `d.pop('clave')` |
| Solo si no existe | No directo | `d.setdefault('k', v)` |
| Claves | `names(l)` | `d.keys()` |
| Valores | `unlist(l)` | `d.values()` |
| Pares | No directo | `d.items()` |
| Iterar | `for (n in names(l))` | `for k, v in d.items():` |
| Pertenencia clave | `'k' %in% names(l)` | `'k' in d` |
| Frecuencias | `table(v)` | `Counter(v)` |
| Transformar | `setNames(sapply(...), ...)` | `{k: f(v) for k, v in d.items()}` |
| Combinar | `modifyList(l1, l2)` | `l1 \| l2` o `{**l1, **l2}` |
| Copia profunda | `rapply(l, identity)` | `copy.deepcopy(d)` |

> **Próximo capítulo:** Entrada del usuario y bucles `while` — Interacción con el usuario, validación de datos y control de flujo con loops mientras se cumple una condición.